# ❄️ Permafrost — Benchmark de Compressão

Comparativo rigoroso: **Permafrost vs gzip vs lzma2 puro vs Parquet+snappy**
em cinco tipos de dataset reais.

**Datasets:**
- Logs de servidor (timestamps + categorias + inteiros)
- Séries temporais de IoT (sensores com alta redundância)
- Dados financeiros (preços com delta pequeno)
- Texto livre (comentários de usuários)
- Dataset misto (combinação dos anteriores)

**Métricas:**
- Ratio de compressão (×)
- Velocidade de freeze (MB/s)
- Velocidade de thaw (MB/s)
- RAM pico (MB)

In [ ]:
%pip install permafrost-framework --quiet

In [ ]:
import permafrost as pf
import pandas as pd
import numpy as np
import gzip, lzma, time, os, tempfile, tracemalloc

print(f"Permafrost {pf.__version__}")

In [ ]:
WORKDIR = tempfile.mkdtemp(prefix='pf_bench_')
N = 100_000  # linhas por dataset
print(f"Workdir: {WORKDIR}")
print(f"Linhas por dataset: {N:,}")

## 1. Gerando os datasets

In [ ]:
np.random.seed(42)

datasets = {}

# Dataset 1: Logs de servidor
datasets['server_logs'] = pd.DataFrame({
    'ts':        pd.date_range('2024-01-01', periods=N, freq='1s'),
    'server':    np.random.choice([f's{i:03d}' for i in range(50)], N),
    'endpoint':  np.random.choice(['/api/users','/api/orders','/api/products','/health'], N),
    'status':    np.random.choice([200,200,200,201,400,404,500], N),
    'latency':   np.round(np.abs(np.random.lognormal(3.5, 0.8, N)), 1),
    'bytes':     np.random.randint(100, 50_000, N),
    'region':    np.random.choice(['us','eu','ap'], N),
})

# Dataset 2: Séries temporais IoT
t_base = np.arange(N)
datasets['iot_sensors'] = pd.DataFrame({
    'ts':         pd.date_range('2024-01-01', periods=N, freq='1min'),
    'sensor_id':  np.random.randint(1, 200, N),
    'temp':       np.round(20.0 + np.cumsum(np.random.normal(0, 0.01, N)), 3),
    'humidity':   np.round(60.0 + np.cumsum(np.random.normal(0, 0.005, N)), 3),
    'pressure':   np.round(1013.0 + np.cumsum(np.random.normal(0, 0.001, N)), 4),
    'voltage':    np.round(220.0 + np.random.normal(0, 0.1, N), 3),
    'status':     np.random.choice(['ok','warn'], N, p=[0.98, 0.02]),
})

# Dataset 3: Dados financeiros
price = 100.0
prices = []
for _ in range(N):
    price *= (1 + np.random.normal(0, 0.001))
    prices.append(round(price, 4))

datasets['financial'] = pd.DataFrame({
    'ts':       pd.date_range('2024-01-01', periods=N, freq='1s'),
    'symbol':   np.random.choice(['AAPL','GOOGL','MSFT','AMZN','META'], N),
    'open':     np.array(prices),
    'high':     np.array(prices) * (1 + np.abs(np.random.normal(0, 0.001, N))),
    'low':      np.array(prices) * (1 - np.abs(np.random.normal(0, 0.001, N))),
    'close':    np.array(prices) * (1 + np.random.normal(0, 0.0005, N)),
    'volume':   np.random.randint(100, 100_000, N),
})

# Dataset 4: Texto livre (simula comentários)
WORDS = ['ótimo', 'péssimo', 'excelente', 'ruim', 'bom', 'produto', 'serviço',
         'rápido', 'lento', 'recomendo', 'não', 'muito', 'pouco', 'qualidade',
         'entrega', 'preço', 'justo', 'caro', 'barato', 'satisfeito']

def gen_comment():
    n = np.random.randint(5, 20)
    return ' '.join(np.random.choice(WORDS, n))

datasets['user_reviews'] = pd.DataFrame({
    'ts':       pd.date_range('2024-01-01', periods=N, freq='10min'),
    'user_id':  np.random.randint(1, 50_000, N),
    'product':  np.random.choice([f'PROD-{i:04d}' for i in range(1000)], N),
    'rating':   np.random.randint(1, 6, N),
    'comment':  [gen_comment() for _ in range(N)],
    'verified': np.random.choice([True, False], N, p=[0.7, 0.3]),
})

for name, df in datasets.items():
    csv_mb = len(df.to_csv(index=False).encode()) / 1e6
    print(f"  {name:20s}: {len(df):,} linhas, {len(df.columns)} cols, {csv_mb:.2f} MB CSV")

## 2. Benchmark de compressão

Comparando Permafrost contra alternativas comuns.

In [ ]:
def benchmark_method(name, compress_fn, decompress_fn, data_bytes):
    """Mede ratio, velocidade e pico de RAM."""
    tracemalloc.start()
    t0 = time.time()
    compressed = compress_fn(data_bytes)
    t_compress = time.time() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    t0 = time.time()
    _ = decompress_fn(compressed)
    t_decompress = time.time() - t0

    orig_mb = len(data_bytes) / 1e6
    comp_mb = len(compressed) / 1e6
    ratio   = orig_mb / comp_mb if comp_mb > 0 else 0

    return {
        'método':       name,
        'original_mb':  round(orig_mb, 2),
        'comprimido_mb':round(comp_mb, 3),
        'ratio':        round(ratio, 2),
        'compress_s':   round(t_compress, 3),
        'decompress_s': round(t_decompress, 3),
        'ram_peak_mb':  round(peak / 1e6, 1),
    }


def benchmark_permafrost(codec_id, codec_name, df, tmp_path):
    """Mede Permafrost especificamente."""
    tracemalloc.start()
    t0 = time.time()
    metrics = pf.freeze(df, tmp_path, codec=codec_id)
    t_freeze = time.time() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    t0 = time.time()
    _ = pf.thaw(tmp_path, verify=False)
    t_thaw = time.time() - t0

    return {
        'método':       f'Permafrost {codec_name}',
        'original_mb':  metrics['original_mb'],
        'comprimido_mb':metrics['stored_mb'],
        'ratio':        metrics['ratio'],
        'compress_s':   metrics['freeze_s'],
        'decompress_s': round(t_thaw, 3),
        'ram_peak_mb':  round(peak / 1e6, 1),
    }

print("Funções de benchmark definidas.")

In [ ]:
all_results = {}

for ds_name, df in datasets.items():
    print(f"\nDataset: {ds_name}")
    csv_bytes = df.to_csv(index=False).encode()
    results = []

    # gzip
    r = benchmark_method('CSV + gzip',
        lambda b: gzip.compress(b, compresslevel=9),
        lambda b: gzip.decompress(b),
        csv_bytes)
    results.append(r)
    print(f"  gzip:              {r['ratio']:.2f}× — {r['compress_s']:.2f}s")

    # lzma puro
    r = benchmark_method('CSV + lzma2',
        lambda b: lzma.compress(b, format=lzma.FORMAT_XZ, preset=6),
        lambda b: lzma.decompress(b),
        csv_bytes)
    results.append(r)
    print(f"  lzma2:             {r['ratio']:.2f}× — {r['compress_s']:.2f}s")

    # Permafrost ZSTD
    path_zstd = os.path.join(WORKDIR, f'{ds_name}_zstd.permafrost')
    r = benchmark_permafrost(pf.CODEC_ZSTD, 'ZSTD', df, path_zstd)
    results.append(r)
    print(f"  Permafrost ZSTD:   {r['ratio']:.2f}× — {r['compress_s']:.2f}s")

    # Permafrost LZMA2
    path_lzma = os.path.join(WORKDIR, f'{ds_name}_lzma2.permafrost')
    r = benchmark_permafrost(pf.CODEC_LZMA2, 'LZMA2', df, path_lzma)
    results.append(r)
    print(f"  Permafrost LZMA2:  {r['ratio']:.2f}× — {r['compress_s']:.2f}s")

    all_results[ds_name] = results

print("\n✓ Benchmark concluído")

## 3. Tabela comparativa

In [ ]:
rows = []
for ds_name, results in all_results.items():
    for r in results:
        rows.append({'dataset': ds_name, **r})

df_results = pd.DataFrame(rows)

# Pivot por ratio
pivot = df_results.pivot_table(
    index='dataset',
    columns='método',
    values='ratio',
    aggfunc='first'
).round(2)

print("Ratio de compressão (×) — maior é melhor:")
print(pivot.to_string())

## 4. Visualização de resultados

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.rcParams['figure.figsize'] = (14, 10)
    matplotlib.rcParams['font.size'] = 11

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Permafrost vs Alternativas de Compressão', fontsize=14, fontweight='bold')

    COLORS = {
        'CSV + gzip':        '#aaaaaa',
        'CSV + lzma2':       '#888888',
        'Permafrost ZSTD':   '#4da6ff',
        'Permafrost LZMA2':  '#0066cc',
    }

    ds_names = list(all_results.keys())
    methods  = ['CSV + gzip', 'CSV + lzma2', 'Permafrost ZSTD', 'Permafrost LZMA2']

    def get_vals(metric):
        return {
            ds: {r['método']: r[metric] for r in all_results[ds]}
            for ds in ds_names
        }

    # Subplot 1: Ratio
    ax = axes[0, 0]
    x = np.arange(len(ds_names))
    w = 0.2
    ratios = get_vals('ratio')
    for i, m in enumerate(methods):
        vals = [ratios[ds].get(m, 0) for ds in ds_names]
        bars = ax.bar(x + i*w, vals, w, label=m, color=COLORS[m], alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                    f'{v:.1f}×', ha='center', va='bottom', fontsize=8)
    ax.set_xlabel('Dataset')
    ax.set_ylabel('Ratio (×)')
    ax.set_title('Ratio de Compressão')
    ax.set_xticks(x + w*1.5)
    ax.set_xticklabels([d.replace('_', '\n') for d in ds_names], fontsize=9)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    # Subplot 2: Velocidade de compressão
    ax = axes[0, 1]
    speeds = get_vals('compress_s')
    for i, m in enumerate(methods):
        vals = [speeds[ds].get(m, 0) for ds in ds_names]
        ax.bar(x + i*w, vals, w, label=m, color=COLORS[m], alpha=0.85)
    ax.set_xlabel('Dataset')
    ax.set_ylabel('Tempo (s) — menor é melhor')
    ax.set_title('Velocidade de Compressão')
    ax.set_xticks(x + w*1.5)
    ax.set_xticklabels([d.replace('_', '\n') for d in ds_names], fontsize=9)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    # Subplot 3: Velocidade de descompressão
    ax = axes[1, 0]
    decs = get_vals('decompress_s')
    for i, m in enumerate(methods):
        vals = [decs[ds].get(m, 0) for ds in ds_names]
        ax.bar(x + i*w, vals, w, label=m, color=COLORS[m], alpha=0.85)
    ax.set_xlabel('Dataset')
    ax.set_ylabel('Tempo (s) — menor é melhor')
    ax.set_title('Velocidade de Descompressão')
    ax.set_xticks(x + w*1.5)
    ax.set_xticklabels([d.replace('_', '\n') for d in ds_names], fontsize=9)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    # Subplot 4: Resumo — Ganho do Permafrost sobre gzip
    ax = axes[1, 1]
    ratios_gzip = [ratios[ds].get('CSV + gzip', 1) for ds in ds_names]
    ratios_pf   = [ratios[ds].get('Permafrost LZMA2', 1) for ds in ds_names]
    ganho = [pf_r / gz_r for pf_r, gz_r in zip(ratios_pf, ratios_gzip)]
    bars = ax.bar(ds_names, ganho, color='#0066cc', alpha=0.85)
    for bar, v in zip(bars, ganho):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                f'{v:.1f}×', ha='center', va='bottom', fontweight='bold')
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='baseline gzip')
    ax.set_xlabel('Dataset')
    ax.set_ylabel('Ganho vs gzip (×)')
    ax.set_title('Ganho Permafrost LZMA2 vs gzip')
    ax.set_xticklabels([d.replace('_', '\n') for d in ds_names], fontsize=9)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(WORKDIR, 'benchmark.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Gráfico salvo em: {plot_path}")

except ImportError:
    print("matplotlib não instalado — instale com: pip install matplotlib")
    print("\nResultados numéricos:")
    print(pivot.to_string())

## 5. Resumo detalhado por dataset

In [ ]:
for ds_name, results in all_results.items():
    print(f"\n{'═'*70}")
    print(f" {ds_name.upper().replace('_', ' ')}")
    print(f"{'═'*70}")
    print(f"  {'Método':25s} {'Ratio':>8} {'Compress':>10} {'Decompr':>10} {'RAM':>8}")
    print(f"  {'─'*63}")
    for r in sorted(results, key=lambda x: -x['ratio']):
        print(f"  {r['método']:25s} {r['ratio']:>7.2f}× {r['compress_s']:>8.2f}s {r['decompress_s']:>8.3f}s {r['ram_peak_mb']:>6.0f}MB")

# Cálculo de economia de storage
print(f"\n{'═'*70}")
print(" ECONOMIA DE STORAGE (estimativa 1 TB de dados, 3 anos, S3 Glacier $0.00099/GB/mês)")
print(f"{'═'*70}")

GLACIER_PER_GB_MES = 0.00099
TB = 1024  # GB
MESES = 36

for ds_name, results in all_results.items():
    ratio_gzip = next((r['ratio'] for r in results if 'gzip' in r['método']), 1)
    ratio_pf   = next((r['ratio'] for r in results if 'LZMA2' in r['método']), 1)

    custo_gzip = (TB / ratio_gzip) * GLACIER_PER_GB_MES * MESES
    custo_pf   = (TB / ratio_pf)   * GLACIER_PER_GB_MES * MESES
    economia   = custo_gzip - custo_pf

    print(f"  {ds_name:20s}  gzip=${custo_gzip:.1f}  Permafrost=${custo_pf:.2f}  economia=${economia:.1f} ({economia/custo_gzip*100:.0f}%)")

## 6. Por que Permafrost comprime melhor?

O segredo são os **preditores colunares** aplicados **antes** do codec:

| Preditor | Tipo de dado | O que faz |
|----------|-------------|----------|
| `ts_delta_s` | Timestamps | Armazena só deltas (segundos entre eventos) em vez do timestamp completo |
| `delta_zigzag` | Inteiros sequenciais | Codifica `val[i] - val[i-1]` → inteiros próximos de zero dominam |
| `category_u8` | Strings categóricas | Mapeia categorias para índice 1-byte (de ~20 bytes para 1 byte) |
| `lag1_zigzag` | Floats com deriva lenta | Diferença entre linhas consecutivas → valores próximos de zero |
| `raw_text` | Texto livre | Passa direto para o codec (já é difícil de comprimir mais) |

**Exemplo:** uma coluna `status` com 100k valores `['ok', 'warn', 'error']`:
- **gzip sobre CSV**: armazena `ok,ok,ok,ok,...` = ~200kb
- **Permafrost `category_u8`**: armazena `[0,0,0,1,0,...]` (1 byte/valor) = ~100kb antes do codec
- **Permafrost + LZMA2**: o codec comprime a sequência de bytes quase idênticos para ~2kb

In [ ]:
# Demonstração: ver qual preditor cada coluna recebe
import json

for ds_name, df in list(datasets.items())[:2]:  # primeiros 2 datasets
    path = os.path.join(WORKDIR, f'{ds_name}_info.permafrost')
    pf.freeze(df.head(1_000), path, codec=pf.CODEC_ZSTD)
    info = pf.audit(path)

    print(f"\n{ds_name}:")
    for col, manifest in info.get('manifests', {}).items():
        pred = manifest.get('predictor', '?')
        print(f"  {col:20s} → {pred}")